# **Chapter 07. Training Pipeline**

* 訓練模型所需的大部分時間都花在`攝取(ingest)`資料上, 也就是`讀取資料並將其轉換為模型可用的形式`
* 如果能在這個階段做出簡化與加速, 就可以增加訓練效率, 方法有:
    * 有效率的儲存方式: 以易讀取的方式儲存前置處理後的值
    * 平行讀取資料: 在攝取資料時, 儲存設備的速度往往成為 bottleneck, 因此平行讀取資料會有很大的幫助
    * 在訓練的同時準備影像: 盡可能在 CPU 上對影像進行前置處理, 同時在 GPU 上進行訓練
    * 最大化 GPU 使用率: 盡可能在 GPU 上進行矩陣與數學運算 (如果前置處理時混涉及這類運算, 那就考慮把它推上 GPU)

### 有效率的儲存方式
* 為什麼 TensorFlow Record 是高效率的儲存機制？
    * 與逐個檔案讀取相比, TFRecord 透過單一網路連結來讀取影像, 且一次讀取一批, 減少GPU等待下一批影像的時間, 而不是為每個檔案開啟一個連結 (開檔時間過長, 每次 open() 都是 system call)
    * 檔案大小在 10MB~100MB 之間, 能夠平衡從多個 worker 讀取影像的能力, 且每個檔案需要打開足夠長的時間, 以分攤讀取多個批次的第一個位元組的延遲 -> 如果一個 TFRecord 檔太小，開檔開銷就會變多（又回到小檔案問題）; 如果太大（幾 GB），又會讓分散式訓練（multi-worker）難以平行載入，因為一個 worker 可能卡在讀取一大檔上。
    * 從檔案讀取的位元組可以立即映射到記憶體結構, 無需剖析檔案或處理不同類型機器之間的儲存佈局差異

* 先前幾章, 我們對JPEG影像的處理過程都是: decode -> 壓縮到[0,1]之間 -> Flatten -> 將這個陣列寫至 tf_record; 但其實 該執行什麼都是`效率與重複利用性之間的取捨`
* 例如: 在寫出tf_record之前的處理愈多, 在訓練生產線中需要執行的處理就愈少, 但可重複利用性就會下降 (記得ch6說的進模型前共用函式 vs. 模型內共用前置處理 的對比)

### 平行讀取資料

In [ ]:
import tensorflow as tf
from tensorflow.data.experimental import AUTO
from tensorflow.data.experimental import AUTOTUNE

class _Preprocessor:    
    def __init__(self):
        # 初始化時不用做事
        pass
    
    def read_from_tfr(self, proto):
        # 這裡是訓練資料 (tf record) 的讀取方式
        feature_description = {
            'image': tf.io.VarLenFeature(tf.float32),
            'shape': tf.io.VarLenFeature(tf.int64),
            'label': tf.io.FixedLenFeature([], tf.string, default_value=''),
            'label_int': tf.io.FixedLenFeature([], tf.int64, default_value=0),
        }
        rec = tf.io.parse_single_example(
            proto, feature_description
        )
        shape = tf.sparse.to_dense(rec['shape'])
        img = tf.reshape(tf.sparse.to_dense(rec['image']), shape)
        label_int = rec['label_int']
        return img, label_int
    
    def read_from_jpegfile(self, filename):
        # 這裡是預測資料 (jpeg image) 的讀取方式: 讀取 -> 解碼 -> 縮放至 [0, 1]
        img = tf.io.read_file(filename)
        img = tf.image.decode_jpeg(img, channels=IMG_CHANNELS)
        img = tf.image.convert_image_dtype(img, tf.float32)
        return img
      
    def preprocess(self, img):
        # 進到模型前的 preprocessing 再將圖片 resize
        return tf.image.resize_with_pad(img, 2*IMG_HEIGHT, 2*IMG_WIDTH)

# 原先在 ch6 的寫法:
# 1. 建立一個 TF Record Dataset
# 2. 將每一個 record 傳給 read_from_tfr, 拿回 (image, label) 的 metadata
# 3. 使用 _preproc_img_label 來進行前置處理
def create_preproc_dataset_prev(pattern):
    preproc = _Preprocessor()
    trainds = tf.data.TFRecordDataset(pattern) \
                .map(preproc.read_from_tfr) \
                .map(lambda img, label: (preproc.preprocess(img), label))

# 可以這樣加速:
# 1. 讓 TensorFlow 自動使用多顆 CPU 平行讀取
# 2. 也讓 map 運算平行化
def create_preproc_dataset_new(pattern):
    preproc = _Preprocessor()
    trainds = tf.data.TFRecordDataset(pattern, num_parallel_reads=AUTO) \
                .map(preproc.read_from_tfr, num_parallel_calls=AUTOTUNE) \
                .map(lambda img, label: (preproc.preprocess(img), label))